In [ ]:
!pip install -q ultralytics imagehash pillow pandas scikit-learn pyyaml

import os
import shutil
import hashlib
from pathlib import Path

WORK = Path("/content/ppe_project")
RAW = WORK / "raw"
DATASET = WORK / "dataset"

WORK.mkdir(parents=True, exist_ok=True)
RAW.mkdir(parents=True, exist_ok=True)
DATASET.mkdir(parents=True, exist_ok=True)

print("WORK:", WORK)
print("RAW:", RAW)
print("DATASET:", DATASET)


## 1) Download Construction Site Safety dataset from Roboflow

In [ ]:
!pip install -q roboflow

from getpass import getpass
from roboflow import Roboflow

ROBOFLOW_API_KEY = getpass("Enter your Roboflow API Key: ")

rf = Roboflow(api_key=ROBOFLOW_API_KEY)

print("Roboflow connected successfully")

workspace = rf.workspace()

print("Workspace connected:", workspace)


In [ ]:
from roboflow import Roboflow

project = rf.workspace("roboflow-universe-projects").project(
    "construction-site-safety"
)

dataset = project.version(27).download(
    "yolov8",
    location="/content/ppe_project/raw/construction_safety"
)

print("Downloaded successfully")
print("Location:", dataset.location)


In [ ]:
from pathlib import Path
import yaml

CSS = Path("/content/ppe_project/raw/construction_safety")

print("=" * 50)
print("DATASET STRUCTURE")
print("=" * 50)

for item in CSS.iterdir():
    print(item.name, "->", "DIR" if item.is_dir() else "FILE")

print()
print("=" * 50)
print("DATA.YAML")
print("=" * 50)

yaml_path = CSS / "data.yaml"

with open(yaml_path, "r") as f:
    css_yaml = yaml.safe_load(f)

print("Classes:")
for i, name in enumerate(css_yaml["names"]):
    print(i, "->", name)


DATASET STRUCTURE
valid -> DIR
test -> DIR
data.yaml -> FILE
README.dataset.txt -> FILE
README.roboflow.txt -> FILE
train -> DIR

DATA.YAML
Classes:
0 -> Hardhat
1 -> Mask
2 -> NO-Hardhat
3 -> NO-Mask
4 -> NO-Safety Vest
5 -> Person
6 -> Safety Cone
7 -> Safety Vest
8 -> machinery
9 -> vehicle


## 2) Download PPE Detection v1 dataset from Kaggle

In [ ]:
!pip install -q kaggle

from google.colab import files
files.upload()

import os
import shutil

os.makedirs("/root/.kaggle", exist_ok=True)

shutil.copy(
    "/content/kaggle.json",
    "/root/.kaggle/kaggle.json"
)

os.chmod("/root/.kaggle/kaggle.json", 0o600)

print("Kaggle credentials configured successfully")


In [ ]:
import os
from pathlib import Path

PPE_DIR = Path("/content/ppe_project/raw/ppe_detection")
PPE_DIR.mkdir(parents=True, exist_ok=True)

!kaggle datasets download -d beyzakucuk/ppe-detection-v1 -p /content/ppe_project/raw/ppe_detection


In [ ]:
import zipfile
from pathlib import Path

zip_path = Path("/content/ppe_project/raw/ppe_detection/ppe-detection-v1.zip")
extract_dir = Path("/content/ppe_project/raw/ppe_detection/data")

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_dir)

print("Extracted successfully")


Extracted successfully


In [ ]:
from pathlib import Path

print("Top-level contents:")
print()

for p in extract_dir.iterdir():
    print("DIR " if p.is_dir() else "FILE", p.name)

print()
print("YAML files:")

for p in extract_dir.rglob("*.yaml"):
    print(p)


Top-level contents:

DIR  valid
DIR  test
FILE data.yaml
FILE README.dataset.txt
FILE README.roboflow.txt
DIR  train

YAML files:
/content/ppe_project/raw/ppe_detection/data/data.yaml


In [ ]:
import yaml

yaml_path = Path("/content/ppe_project/raw/ppe_detection/data/data.yaml")

with open(yaml_path, "r") as f:
    ppe_yaml = yaml.safe_load(f)

print("Classes:")
for i, name in enumerate(ppe_yaml["names"]):
    print(f"{i} -> {name}")


Classes:
0 -> boots
1 -> gloves
2 -> goggles
3 -> helmet
4 -> person
5 -> vest


## 3) Remap PPE_V1 to hardhat / vest / person

In [ ]:
from pathlib import Path

WORK = Path("/content/ppe_project")
RAW = WORK / "raw"
DATASET = WORK / "dataset"

In [ ]:
import shutil
from pathlib import Path

PPE_SRC = extract_dir

PPE_CONVERTED = WORK / "raw" / "ppe_converted"
PPE_CONVERTED.mkdir(parents=True, exist_ok=True)

# 0 boots, 1 gloves, 2 goggles, 3 helmet, 4 person, 5 vest
PPE_TO_FINAL = {3: 0, 4: 4, 5: 2}   # helmet->hardhat, person->person, vest->vest

for split in ["train", "valid", "test"]:
    src_img_dir = PPE_SRC / split / "images"
    src_lbl_dir = PPE_SRC / split / "labels"
    if not src_img_dir.exists():
        print(f"Skipping {split}: no images folder found ({src_img_dir})")
        continue

    dst_img_dir = PPE_CONVERTED / split / "images"
    dst_lbl_dir = PPE_CONVERTED / split / "labels"
    dst_img_dir.mkdir(parents=True, exist_ok=True)
    dst_lbl_dir.mkdir(parents=True, exist_ok=True)

    kept = dropped = boxes_created = 0

    for img_path in src_img_dir.glob("*"):
        lbl_path = src_lbl_dir / f"{img_path.stem}.txt"
        if not lbl_path.exists():
            continue

        new_lines = []
        for line in lbl_path.read_text().splitlines():
            parts = line.split()
            if len(parts) < 7:
                continue
            old_id = int(parts[0])
            if old_id not in PPE_TO_FINAL:
                continue

            coords = list(map(float, parts[1:]))
            if len(coords) < 6 or len(coords) % 2 != 0:
                continue

            xs, ys = coords[0::2], coords[1::2]
            xmin, xmax, ymin, ymax = min(xs), max(xs), min(ys), max(ys)
            xc, yc, w, h = (xmin+xmax)/2, (ymin+ymax)/2, xmax-xmin, ymax-ymin

            new_lines.append(f"{PPE_TO_FINAL[old_id]} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}")
            boxes_created += 1

        if not new_lines:
            dropped += 1
            continue

        shutil.copy2(img_path, dst_img_dir / img_path.name)
        (dst_lbl_dir / lbl_path.name).write_text("\n".join(new_lines))
        kept += 1

    print(f"===== {split.upper()} =====")
    print("Kept:", kept, "| Dropped:", dropped, "| Boxes:", boxes_created)


===== TRAIN =====
Kept: 8482 | Dropped: 5467 | Boxes: 29185
===== VALID =====
Kept: 1068 | Dropped: 601 | Boxes: 3699
===== TEST =====
Kept: 1037 | Dropped: 609 | Boxes: 3799


## 4) Remap CSS to the same 5 classes

In [ ]:
import shutil
from pathlib import Path

CSS_SRC = CSS

CSS_CONVERTED = WORK / "raw" / "css_converted"

CSS_TO_FINAL = {0: 0, 2: 1, 7: 2, 4: 3, 5: 4}  # Hardhat, NO-Hardhat, Safety Vest, NO-Safety Vest, Person

for split in ["train", "valid", "test"]:
    src_img_dir = CSS_SRC / split / "images"
    src_lbl_dir = CSS_SRC / split / "labels"
    if not src_img_dir.exists():
        print(f"Skipping {split}: no images folder found ({src_img_dir})")
        continue

    dst_img_dir = CSS_CONVERTED / split / "images"
    dst_lbl_dir = CSS_CONVERTED / split / "labels"
    dst_img_dir.mkdir(parents=True, exist_ok=True)
    dst_lbl_dir.mkdir(parents=True, exist_ok=True)

    kept = dropped = boxes = 0

    for img_path in src_img_dir.glob("*"):
        lbl_path = src_lbl_dir / f"{img_path.stem}.txt"
        if not lbl_path.exists():
            continue

        new_lines = []
        for line in lbl_path.read_text().splitlines():
            parts = line.split()
            if len(parts) != 5:
                continue
            old_id = int(parts[0])
            if old_id not in CSS_TO_FINAL:
                continue
            parts[0] = str(CSS_TO_FINAL[old_id])
            new_lines.append(" ".join(parts))
            boxes += 1

        if not new_lines:
            dropped += 1
            continue

        shutil.copy2(img_path, dst_img_dir / img_path.name)
        (dst_lbl_dir / lbl_path.name).write_text("\n".join(new_lines))
        kept += 1

    print(f"===== {split.upper()} =====")
    print("Kept:", kept, "| Dropped:", dropped, "| Boxes:", boxes)


===== TRAIN =====
Kept: 2515 | Dropped: 88 | Boxes: 22484
===== VALID =====
Kept: 84 | Dropped: 30 | Boxes: 461
===== TEST =====
Kept: 60 | Dropped: 22 | Boxes: 476


## 5) Merge both sources into one manifest and split 80/10/10 per source

In [ ]:
import pandas as pd
import shutil
from pathlib import Path

FINAL_DATASET = WORK / "dataset"

if FINAL_DATASET.exists():
    shutil.rmtree(FINAL_DATASET)

for split in ["train", "val", "test"]:
    (FINAL_DATASET / split / "images").mkdir(parents=True, exist_ok=True)
    (FINAL_DATASET / split / "labels").mkdir(parents=True, exist_ok=True)


def build_manifest(root, source_name):
    rows = []
    for original_split in ["train", "valid", "test"]:
        img_dir = root / original_split / "images"
        lbl_dir = root / original_split / "labels"
        if not img_dir.exists():
            continue
        for img_path in img_dir.glob("*"):
            lbl_path = lbl_dir / f"{img_path.stem}.txt"
            if lbl_path.exists():
                rows.append({"image_path": str(img_path), "label_path": str(lbl_path), "source": source_name})
    return pd.DataFrame(rows)


ppe_df = build_manifest(PPE_CONVERTED, "PPE_V1")
css_df = build_manifest(CSS_CONVERTED, "Construction_Safety")

manifest = pd.concat([ppe_df, css_df], ignore_index=True)

print("Total images:", len(manifest))
print(manifest["source"].value_counts())


Total images: 13246
source
PPE_V1                 10587
Construction_Safety     2659
Name: count, dtype: int64


In [ ]:
from sklearn.model_selection import train_test_split

train_parts, val_parts, test_parts = [], [], []

for source, group in manifest.groupby("source"):
    train_df_g, temp_df = train_test_split(group, test_size=0.20, random_state=42, shuffle=True)
    val_df_g, test_df_g = train_test_split(temp_df, test_size=0.50, random_state=42, shuffle=True)
    train_parts.append(train_df_g)
    val_parts.append(val_df_g)
    test_parts.append(test_df_g)

train_df = pd.concat(train_parts, ignore_index=True)
val_df = pd.concat(val_parts, ignore_index=True)
test_df = pd.concat(test_parts, ignore_index=True)

print("Train:", len(train_df), "| Val:", len(val_df), "| Test:", len(test_df))


Train: 10596 | Val: 1325 | Test: 1325


## 6) Reduce dataset size using perceptual hashing (instead of random reduction)

We compute a perceptual hash (imagehash.phash) for each PPE_V1 image that has no no-hardhat/no-vest labels, and drop only the visually duplicate copies. Any image containing no-hardhat or no-vest is fully protected regardless of source.

In [ ]:
!pip install -q ImageHash

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 1.3 MB/s eta 0:00:00


In [ ]:
import imagehash
from PIL import Image
from pathlib import Path

In [ ]:

import imagehash
from PIL import Image
from pathlib import Path

TARGET_CLASSES = {1, 3}   # no-hardhat, no-vest

def label_has_target_class(label_path):
    for line in Path(label_path).read_text().splitlines():
        parts = line.split()
        if parts and int(parts[0]) in TARGET_CLASSES:
            return True
    return False

def compute_hash(image_path):
    try:
        return imagehash.phash(Image.open(image_path))
    except Exception:
        return None


train_df["has_no_class"] = train_df["label_path"].apply(label_has_target_class)

protected_df = train_df[train_df["has_no_class"]]                 # fully protected
positive_only_df = train_df[~train_df["has_no_class"]]

ppe_positive_only = positive_only_df[positive_only_df["source"] == "PPE_V1"].copy()
css_positive_only = positive_only_df[positive_only_df["source"] == "Construction_Safety"]

print("Hashing", len(ppe_positive_only), "PPE_V1 images (this may take a few minutes)...")
ppe_positive_only["phash"] = ppe_positive_only["image_path"].apply(compute_hash)
ppe_positive_only = ppe_positive_only.dropna(subset=["phash"])

before = len(ppe_positive_only)
ppe_positive_only_deduped = ppe_positive_only.drop_duplicates(subset=["phash"], keep="first")
after = len(ppe_positive_only_deduped)

print(f"Before dedup: {before} images")
print(f"After dedup : {after} images")
print(f"Detected duplicate ratio: {(before - after) / before * 100:.1f}%")


Hashing 8469 PPE_V1 images (this may take a few minutes)...
Before dedup: 8469 images
After dedup : 5058 images
Detected duplicate ratio: 40.3%


In [ ]:
# If the duplicate ratio is small (under ~15%) and you want an even smaller dataset,
# also set this below 1.0 (removes an extra random fraction of what remains after dedup)
EXTRA_RANDOM_KEEP_FRACTION = 1.0   # 1.0 = no extra reduction, or e.g. 0.7 for extra reduction

ppe_final = ppe_positive_only_deduped.sample(frac=EXTRA_RANDOM_KEEP_FRACTION, random_state=42)

train_df = pd.concat(
    [protected_df, ppe_final, css_positive_only],
    ignore_index=True
)

print("Final train size after dedup:", len(train_df))
print(train_df["source"].value_counts())


Final train size after dedup: 7185
source
PPE_V1                 5058
Construction_Safety    2127
Name: count, dtype: int64


## 7) Copy final files and write data.yaml

In [ ]:
def copy_split(df, split_name):
    img_out = FINAL_DATASET / split_name / "images"
    lbl_out = FINAL_DATASET / split_name / "labels"
    for _, row in df.iterrows():
        src_img = Path(row["image_path"])
        src_lbl = Path(row["label_path"])
        shutil.copy2(src_img, img_out / src_img.name)
        shutil.copy2(src_lbl, lbl_out / src_lbl.name)
    print(split_name.upper(), "->", len(df), "images")


copy_split(train_df, "train")
copy_split(val_df, "val")
copy_split(test_df, "test")


TRAIN -> 7185 images
VAL -> 1325 images
TEST -> 1325 images


In [ ]:
yaml_content = f"""
path: {FINAL_DATASET}

train: train/images
val: val/images
test: test/images

names:
  0: hardhat
  1: no-hardhat
  2: vest
  3: no-vest
  4: person
"""

(FINAL_DATASET / "data.yaml").write_text(yaml_content)
print(yaml_content)



path: /content/ppe_project/dataset

train: train/images
val: val/images
test: test/images

names:
  0: hardhat
  1: no-hardhat
  2: vest
  3: no-vest
  4: person



## 8) Lighter targeted oversampling (N=2) for no-hardhat/no-vest images

In [ ]:
from pathlib import Path

TRAIN_IMG = FINAL_DATASET / "train" / "images"
TRAIN_LBL = FINAL_DATASET / "train" / "labels"

oversample_files = []
for lbl_path in TRAIN_LBL.glob("*.txt"):
    classes_in_file = {int(l.split()[0]) for l in lbl_path.read_text().splitlines() if l.strip()}
    if classes_in_file & {1, 3}:
        oversample_files.append(lbl_path)

print("Images containing no-hardhat/no-vest:", len(oversample_files))
print("Total train images before oversampling:", len(list(TRAIN_IMG.glob("*"))))


Images containing no-hardhat/no-vest: 1747
Total train images before oversampling: 7185


In [ ]:
import shutil

N_EXTRA_COPIES = 2

added = 0
for lbl_path in oversample_files:
    img_candidates = list(TRAIN_IMG.glob(f"{lbl_path.stem}.*"))
    if not img_candidates:
        continue
    img_path = img_candidates[0]

    for copy_idx in range(1, N_EXTRA_COPIES + 1):
        new_stem = f"{lbl_path.stem}_os{copy_idx}"
        shutil.copy2(img_path, TRAIN_IMG / f"{new_stem}{img_path.suffix}")
        shutil.copy2(lbl_path, TRAIN_LBL / f"{new_stem}.txt")
        added += 1

print("Extra copies added:", added)
print("Total train images after oversampling:", len(list(TRAIN_IMG.glob("*"))))


Extra copies added: 3494
Total train images after oversampling: 10679


In [ ]:
from collections import Counter

CLASS_NAMES = {0: "hardhat", 1: "no-hardhat", 2: "vest", 3: "no-vest", 4: "person"}
counter = Counter()

for label_file in TRAIN_LBL.glob("*.txt"):
    for line in label_file.read_text().splitlines():
        parts = line.split()
        if len(parts) == 5:
            counter[int(parts[0])] += 1

print("===== FINAL TRAIN =====")
for cls_id, name in CLASS_NAMES.items():
    print(f"{name:12s}: {counter[cls_id]}")


===== FINAL TRAIN =====
hardhat     : 14259
no-hardhat  : 5901
vest        : 13529
no-vest     : 10059
person      : 25365


In [ ]:
print("Total train images:", len(list(TRAIN_IMG.glob("*"))))

Total train images: 10679


## 9) Training

Note: WORK is currently under /content, which is local to this Colab VM. If the session disconnects or resets during training, the checkpoints in this run will be lost, since /content is not backed by Google Drive. If you want crash protection, mount Google Drive and set WORK (and therefore RUN_PROJECT) to a path under /content/drive/MyDrive before running this cell.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

WORK = Path("/content/drive/MyDrive/ppe_project")
RAW = WORK / "raw"
DATASET = WORK / "dataset"

WORK.mkdir(parents=True, exist_ok=True)
RAW.mkdir(parents=True, exist_ok=True)
DATASET.mkdir(parents=True, exist_ok=True)

Mounted at /content/drive


In [ ]:
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 77.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 7.8 MB/s eta 0:00:00


In [ ]:
RUN_PROJECT = str(WORK / "runs")
RUN_NAME = "yolo11s_adamw_oversampled_imgsz640_smaller"

from ultralytics import YOLO

BASELINE_EPOCH18_RECALL_AGG = 0.615
BASELINE_FINAL_RECALL = {"hardhat": 0.754, "no-hardhat": 0.665, "vest": 0.775, "no-vest": 0.697}
BASELINE_FINAL_GAP_HARDHAT = BASELINE_FINAL_RECALL["hardhat"] - BASELINE_FINAL_RECALL["no-hardhat"]
BASELINE_FINAL_GAP_VEST = BASELINE_FINAL_RECALL["vest"] - BASELINE_FINAL_RECALL["no-vest"]

CHECK_EPOCH = 18
checked = {"done": False}


def progress_report(trainer):
    current_epoch = trainer.epoch + 1
    if current_epoch != CHECK_EPOCH or checked["done"]:
        return
    checked["done"] = True

    try:
        box = trainer.validator.metrics.box
        names = trainer.validator.metrics.names
        class_indices = box.ap_class_index
        recall_by_name = {names[int(idx)]: box.r[i] for i, idx in enumerate(class_indices)}
        precision_by_name = {names[int(idx)]: box.p[i] for i, idx in enumerate(class_indices)}
        agg_recall = trainer.validator.metrics.box.mr
    except Exception as e:
        print("Could not read metrics at this checkpoint:", e)
        return

    print(f"{'='*55}")
    print(f"Progress report at epoch {current_epoch}")
    print(f"{'='*55}")
    for name in recall_by_name:
        print(f"{name:12s} recall: {recall_by_name[name]:.4f}   precision: {precision_by_name[name]:.4f}")

    print(f"Aggregate recall now: {agg_recall:.4f}  (baseline @18: {BASELINE_EPOCH18_RECALL_AGG:.4f})")

    gap_hardhat = recall_by_name.get("hardhat", 0) - recall_by_name.get("no-hardhat", 0)
    gap_vest = recall_by_name.get("vest", 0) - recall_by_name.get("no-vest", 0)
    print(f"Gap hardhat/no-hardhat: {gap_hardhat:.4f} (final baseline: {BASELINE_FINAL_GAP_HARDHAT:.4f})")
    print(f"Gap vest/no-vest      : {gap_vest:.4f} (final baseline: {BASELINE_FINAL_GAP_VEST:.4f})")
    print("Final decision will be made once training finishes or stops due to patience.")


import os
import time

def confirm_save(trainer):
    ckpt_path = Path(RUN_PROJECT) / RUN_NAME / "weights" / "last.pt"
    if ckpt_path.exists():
        mtime = time.strftime('%H:%M:%S', time.localtime(os.path.getmtime(ckpt_path)))
        print(f"last.pt saved - last modified: {mtime}")


MODEL = YOLO("yolo11s.pt")
MODEL.add_callback("on_fit_epoch_end", progress_report)
MODEL.add_callback("on_model_save", confirm_save)

results = MODEL.train(
    data=f"{FINAL_DATASET}/data.yaml",
    epochs=60,
    imgsz=640,
    batch=16,
    device=0,
    patience=15,
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    weight_decay=0.0005,
    warmup_epochs=3,
    mosaic=1.0,
    mixup=0.1,
    degrees=5,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    project=RUN_PROJECT,
    name=RUN_NAME,
    save=True,
    save_period=5,
    plots=True,
    verbose=True
)


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
Ultralytics 8.4.142 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/ppe_project/dataset/data.yaml, degrees=5, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=60, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=to

In [ ]:
from google.colab import files

files.download(
    "/content/drive/MyDrive/ppe_project/runs/yolo11s_adamw_oversampled_imgsz640_smaller/weights/best.pt"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from ultralytics import YOLO

BEST = "/content/drive/MyDrive/ppe_project/runs/yolo11s_adamw_oversampled_imgsz640_smaller/weights/best.pt"
DATA = "/content/ppe_project/dataset/data.yaml"

model = YOLO(BEST)

test_results = model.val(
    data=DATA,
    split="test",
    imgsz=640,
    batch=16,
    device=0,
    plots=True,
    verbose=True
)

Ultralytics 8.4.142 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11s summary (fused): 100 layers, 9,414,735 parameters, 0 gradients, 21.4 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.0±0.0 ms, read: 25.3±13.3 MB/s, size: 62.8 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /content/ppe_project/dataset/test/labels... 1325 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1325/1325 640.0it/s 2.1s
val: /content/ppe_project/dataset/test/images/Aitin3617_jpg.rf.02f382d42c6cab30895fe0031d617478.jpg: 2 duplicate labels removed
val: /content/ppe_project/dataset/test/images/Aitin3617_jpg.rf.22d27049f84ee0deddd8183a2246e8c5.jpg: 2 duplicate labels removed
val: New cache created: /content/ppe_project/dataset/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 83/83 3.6it/s 23.0s
 

In [ ]:
print("\n========== TEST EVALUATION ==========")

print(f"Precision : {test_results.box.mp:.4f}")
print(f"Recall    : {test_results.box.mr:.4f}")
print(f"mAP50     : {test_results.box.map50:.4f}")
print(f"mAP50-95  : {test_results.box.map:.4f}")

print("\n========== PER-CLASS RESULTS ==========")

for i, name in model.names.items():
    print(
        f"{name:12s} | "
        f"mAP50-95: {test_results.box.maps[i]:.4f}"
    )


========== TEST EVALUATION ==========
Precision : 0.7949
Recall    : 0.7581
mAP50     : 0.7979
mAP50-95  : 0.5910

========== PER-CLASS RESULTS ==========
hardhat      | mAP50-95: 0.5480
no-hardhat   | mAP50-95: 0.5876
vest         | mAP50-95: 0.5976
no-vest      | mAP50-95: 0.6384
person       | mAP50-95: 0.5836


In [ ]:
from google.colab import files

uploaded = files.upload()
from ultralytics import YOLO

model = YOLO(
    "/content/drive/MyDrive/ppe_project/runs/yolo11s_adamw_oversampled_imgsz640_smaller/weights/best.pt"
)

results = model.predict(
    source="OIP.webp",
    imgsz=640,
    conf=0.25,
    save=True
)

Saving OIP.webp to OIP (1).webp

image 1/1 /content/OIP.webp: 640x448 1 hardhat, 1 person, 12.0ms
Speed: 1.8ms preprocess, 12.0ms inference, 1.3ms postprocess per image at shape (1, 3, 640, 448)
Results saved to /content/runs/detect/predict
